# Task 3, Step 2: Quality Scores and Answer Aggregation

**Manual step 2.** For each rollout:

$$L_i=\frac{1}{T_i}\sum_t\log p_{i,t}(y_{i,t}),\qquad K_i=\frac{1}{T_i}\sum_t D_{KL}(U\|p_{i,t}),\qquad V_i=\frac{1}{T_i}\sum_t\frac{A^{image}_{i,t}}{A^{image}_{i,t}+A^{prompt}_{i,t}+A^{generated}_{i,t}}$$

VisConf pools token KL values across the **current candidates only**, calibrates with their 5th and 95th percentiles, and sets $Z_i=V_iC_i$. Rank-weighted consensus gives $w_i=(N-r_i)/(N-1)$ and returns $\arg\max_a\sum_{i:a_i=a}w_i$.

Keeping the score fixed while changing aggregation isolates the value of consensus. Keeping aggregation fixed while changing the score isolates the value of the quality signal.

Budgets $N\in\{1,2,4,8,16,32\}$, ten seeded orders per question, nested prefixes. Outcomes are averaged over orders within a question, then across questions. Figures use the 800 held-out questions.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task3_test_time_compute")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task3_test_time_compute.src.notebook_utils import *
setup_notebook()

In [ ]:
from task3_test_time_compute.src.evaluate import make_orders, run_offline_budget_evaluation, compare_methods
results = {}
for dataset in DATASETS:
    path = f"{RESULTS_DIR}/offline_{dataset}.csv"
    if os.path.exists(path):
        results[dataset] = pd.read_csv(path)
    else:
        _, test, _ = split_questions(dataset)
        print(f"Evaluating {dataset}: {len(test)} questions x 10 orders x 6 budgets...")
        results[dataset] = run_offline_budget_evaluation(test, make_orders(test))
        results[dataset].to_csv(path, index=False)
    print(dataset, results[dataset].shape)

## 2.1 Accuracy as candidates increase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    summary = results[dataset].groupby(["N", "method"])["accuracy"].mean().mul(100).reset_index()
    coverage = pd.read_csv(f"{RESULTS_DIR}/coverage_{dataset}.csv").groupby("N")["oracle_coverage"].mean().mul(100)
    ax.plot(coverage.index, coverage.values, color="black", linestyle=":", linewidth=2, label="Oracle coverage")
    for method, group in summary.groupby("method"):
        ax.plot(group["N"], group["accuracy"], marker="o", markersize=4, label=METHOD_LABELS[method], color=METHOD_COLORS[method])
    ax.set(title=label, xlabel="Candidates N (log scale)", ylabel="Accuracy (%)", xscale="log")
    ax.set_xticks([1, 2, 4, 8, 16, 32]); ax.set_xticklabels([1, 2, 4, 8, 16, 32])
    ax.legend(fontsize=7.5, ncol=2)
save_figure("task3_2_accuracy_curves")

## 2.2 Every method at every budget

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={"width_ratios": [5, 11]})
tables = {}
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    table = results[dataset].groupby(["method", "N"])["accuracy"].mean().mul(100).unstack("N")
    table = table.loc[[m for m in METHOD_LABELS if m in table.index]]
    tables[dataset] = table
    table.to_csv(f"{RESULTS_DIR}/method_table_{dataset}.csv")
    sns.heatmap(table.rename(index=METHOD_LABELS), annot=True, fmt=".1f", cmap="viridis", ax=ax, cbar=False)
    ax.set(title=f"{label}: accuracy (%)", xlabel="Candidates N", ylabel="")
save_figure("task3_2_method_heatmap")

## 2.3 Main accuracy differences at $N=32$
1,000 paired question-level bootstrap resamples, after averaging over orders. An interval crossing zero means the data do not separate the two methods at this sample size.

In [ ]:
comparisons = [("self_consistency", "first_ans"), ("self_certainty", "first_ans"), ("self_certainty_rwc", "self_certainty"), ("self_certainty_rwc", "self_consistency"), ("mean_logp", "self_certainty")]
vlm_comparisons = comparisons + [("visconf", "vision_only"), ("visconf_rwc", "visconf"), ("visconf_rwc", "self_consistency"), ("visconf_rwc", "self_certainty_rwc"), ("visconf_token_rwc", "visconf_rwc")]
frames = []
for dataset, comps in [("math", comparisons), ("mathvista", vlm_comparisons)]:
    frames.append(compare_methods(results[dataset], comps).assign(dataset=DATASETS[dataset]))
differences = pd.concat(frames, ignore_index=True)
differences.to_csv(f"{RESULTS_DIR}/method_differences_bootstrap.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 0.42 * len(differences) + 1.5))
labels = [f"{DATASETS[d].split(' ')[0]}: {METHOD_LABELS[a]} − {METHOD_LABELS[b]}" for d, a, b in zip(["math"] * len(frames[0]) + ["mathvista"] * len(frames[1]), differences["method_a"], differences["method_b"])]
y = np.arange(len(differences))[::-1]
ax.errorbar(differences["difference_pp"], y, xerr=[differences["difference_pp"] - differences["ci_low"], differences["ci_high"] - differences["difference_pp"]], fmt="o", color="#2c3e50", capsize=3)
ax.axvline(0, color="crimson", linestyle="--")
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
ax.set(title="Accuracy differences at N = 32 with 95% bootstrap intervals", xlabel="Percentage points")
save_figure("task3_2_bootstrap_forest")
display(differences.round(2))

## 2.4 Accuracy against generated tokens
FirstAns consumes one candidate. Token counts are an offline proxy: they exclude prefill, attention collection, and scoring.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
focus = ["first_ans", "self_consistency", "self_certainty", "self_certainty_rwc", "visconf", "visconf_rwc"]
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    summary = results[dataset].groupby(["N", "method"])[["accuracy", "token_cost"]].mean().reset_index()
    for method in focus:
        group = summary[summary["method"] == method]
        if len(group):
            ax.plot(group["token_cost"], group["accuracy"] * 100, marker="o", label=METHOD_LABELS[method], color=METHOD_COLORS[method])
    ax.set(title=label, xlabel="Mean generated tokens per question (log scale)", ylabel="Accuracy (%)", xscale="log")
    ax.legend(fontsize=8)
save_figure("task3_2_accuracy_vs_tokens")